# 01 — Data Loading

Загрузка данных из Excel в DB через ExcelLoader.

**Workflow:**
1. Проверить Excel файл
2. Загрузить через ExcelLoader
3. Верифицировать данные в DB


In [ ]:
# ── Setup ──────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path('.').resolve()
for _p in [ROOT] + list(ROOT.parents):
    if (_p / 'engine').is_dir():
        ROOT = _p; break
sys.path.insert(0, str(ROOT))

COMPANY = None
for _p in [pathlib.Path('.').resolve()] + list(pathlib.Path('.').resolve().parents):
    if _p.parent.name == 'companies':
        COMPANY = _p.name; break
if not COMPANY: COMPANY = 'test_metals'  # fallback
print(f'Company: {COMPANY}, ROOT: {ROOT}')


## 1. Загрузка Excel → DB

In [ ]:
from pathlib import Path
from engine.loader.excel import ExcelLoader
from engine.database.repository import Repository
from engine import DB_PATH

# Path to unified Excel file
excel_path = ROOT / 'companies' / COMPANY / 'data' / 'excel' / f'{COMPANY}_unified_complete.xlsx'
# Alternative: look for any xlsx in data/excel/
if not excel_path.exists():
    excel_dir = ROOT / 'companies' / COMPANY / 'data' / 'excel'
    xlsx_files = list(excel_dir.glob('*.xlsx')) if excel_dir.exists() else []
    excel_dir2 = ROOT / 'companies' / COMPANY / 'data'
    xlsx_files += list(excel_dir2.glob('*unified*.xlsx'))
    if xlsx_files:
        excel_path = xlsx_files[0]
        print(f'Found: {excel_path.name}')
    else:
        print(f'ERROR: No Excel file found in {excel_dir}')

print(f'Excel: {excel_path}')
print(f'Exists: {excel_path.exists()}')


In [ ]:
# Load Excel → DB
with Repository(DB_PATH) as repo:
    loader = ExcelLoader(
        company_id=COMPANY,
        repo=repo,
        db_unit='USD',
        input_default_unit='mUSD',  # Excel values in millions
    )
    result = loader.load(excel_path)

print(f'Rows written: {result.rows_written}')
print(f'Errors: {result.errors}')
print(f'Warnings: {result.warnings}')
print(f'Missing required: {result.missing_required}')


## 2. Verification

In [ ]:
import sqlite3
conn = sqlite3.connect(str(DB_PATH))

print(f'=== {COMPANY} data in DB ===')
for stmt in ['is', 'bs', 'cf']:
    r = conn.execute(f'''
        SELECT COUNT(*) n, MIN(p.year) y0, MAX(p.year) y1
        FROM history_{stmt} h JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=?
    ''', (COMPANY,)).fetchone()
    print(f'  {stmt.upper()}: {r[0]} rows ({r[1]}-{r[2]})')

r = conn.execute('SELECT COUNT(*), SUM(opening_balance) FROM debt_instruments WHERE company_id=?', (COMPANY,)).fetchone()
print(f'  DI: {r[0]} instruments, ${(r[1] or 0)/1e9:.2f}B')

# BS identity check
print(f'\n=== BS Identity ===')
for yr_row in conn.execute(f'''
    SELECT p.year FROM periods p
    WHERE p.company_id=? AND p.year IN (
        SELECT MAX(p2.year) FROM periods p2
        JOIN history_bs h ON h.period_id=p2.period_id
        WHERE h.company_id=?
    )
''', (COMPANY, COMPANY)).fetchall():
    yr = yr_row[0]
    metrics = {}
    for r in conn.execute(f'''
        SELECT h.metric, h.value FROM history_bs h
        JOIN periods p ON h.period_id=p.period_id
        WHERE h.company_id=? AND p.year=?
    ''', (COMPANY, yr)).fetchall():
        metrics[r[0]] = r[1]
    ta = metrics.get('total_assets', 0)
    tl = metrics.get('total_liabilities', 0)
    te = metrics.get('total_equity', 0)
    print(f'  {yr}: A=${ta/1e6:,.0f}M L=${tl/1e6:,.0f}M E=${te/1e6:,.0f}M A-L-E=${(ta-tl-te)/1e6:+,.0f}M')

conn.close()
